> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验一：RMSNorm基础版算子开发与验证


建议学时：1学时


# 实验任务


## 任务描述


本实验在昇腾910B4上开发并验证Qwen2.5使用的RMSNorm基础版算子。实验从模型需求出发，说明该算子的作用、计算结构、开发过程和正确性验证方法。


## 学习目标


了解RMSNorm在大语言模型中的作用和计算结构；掌握将计算公式拆分为主机侧参数准备、设备侧内核计算、框架调用与测试验证的开发流程。


# 任务准备


## 算子定义与接口约定


输入张量的最后一维表示隐藏特征长度，权重为同长度的一维单精度浮点张量，稳定项为正数。将前导维展平为若干行后，算子对每一行独立计算：


RMSNorm不减去均值，而是根据一行数据的均方根进行缩放；同一组可学习权重作用于所有行。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">当前工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">input、weight、output均为float32</td>
</tr>
<tr>
<td style="text-align:left;">输入布局</td>
<td style="text-align:left;">连续Tensor；最后一维必须等于weight.numel()</td>
</tr>
<tr>
<td style="text-align:left;">行数</td>
<td style="text-align:left;">rows = input.numel() / hidden，支持一维、二维及更高维输入</td>
</tr>
<tr>
<td style="text-align:left;">并行划分</td>
<td style="text-align:left;">coreNum = min(8, rows)，rowsPerCore向上取整</td>
</tr>
<tr>
<td style="text-align:left;">标准测试形状</td>
<td style="text-align:left;">rows=128，hidden=1024，eps=1e-6，blockDim=8</td>
</tr>
</tbody></table>


## RMSNorm的作用与计算过程


RMSNorm的输入是模型中间的隐藏状态。它先计算一行数据的均方值，再得到缩放系数，最后与可学习权重逐元素相乘。与需要计算均值的LayerNorm相比，RMSNorm的计算更简洁，适合在深层Transformer中反复使用。


在Qwen2.5中，RMSNorm位于注意力模块和前馈网络等核心计算之前，用来将输入的数值尺度保持在稳定范围。它不改变张量形状，只调整每个位置上隐藏特征的幅值，因此能够自然地嵌入残差连接之间。


## 本实验的算子开发过程


开发过程分为四步：首先根据模型中的计算公式确定输入、输出和权重约定；其次在主机侧根据输入形状计算每个计算核心负责的行数，并将这些参数传给设备；然后在设备侧逐行完成平方和、缩放和权重相乘；最后将算子注册到框架中，用参考实现和模型调用共同验证结果。


图1  RMSNorm的逐行计算流程


## 实验环境准备


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置/说明</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">硬件</td>
<td style="text-align:left;">Ascend 910B4宿主NPU</td>
</tr>
<tr>
<td style="text-align:left;">工具链</td>
<td style="text-align:left;">CANN 8.5.0；需先source set_env.sh</td>
</tr>
<tr>
<td style="text-align:left;">框架接口</td>
<td style="text-align:left;">PyTorch C++ extension，torch.ops.rmsnorm_custom.rms_norm</td>
</tr>
<tr>
<td style="text-align:left;">构建结果</td>
<td style="text-align:left;">out/lib/libascendc_kernels_npu.so、librmsnorm_torch_register.so；out/bin/rmsnorm_*_standalone</td>
</tr>
<tr>
<td style="text-align:left;">测试参考</td>
<td style="text-align:left;">PyTorch torch.rsqrt(input.pow(2).mean(...)+eps)与standalone C++ reference</td>
</tr>
</tbody></table>


# 任务实施


本环节按照“规格与分块—设备端计算—框架接入—构建加载—正确性验证—性能计时与结果分析”六步推进。每一步均对应一个可检查产物：分块参数、核函数、注册入口、构建产物、误差结果和设备侧计时结果。


## 步骤一：定义I/O规格、分块参数与行级任务


本步完成输入、权重、输出和稳定项的接口约定，并形成可传入设备端的Tiling参数与按行分工方案。


主机侧将运行时信息整理为分块参数，其中包括行数、隐藏特征长度、计算核心数量、每个核心负责的行数和稳定项。每个计算核心处理连续的若干行，最后一个核心按实际剩余行数结束，从而覆盖边界数据。


```text
#pragma pack(push, 1)
```


```cpp
struct RmsNormBaselineTiling {
```


```text
uint32_t rows; uint32_t hidden;
```


```text
uint32_t coreNum; uint32_t rowsPerCore;
```


```text
float eps; float invHidden;
```


```text
};
```


```text
#pragma pack(pop)
```


```text
const uint32_t rowBegin = coreId * rowsPerCore_;
```


```text
uint32_t rowEnd = rowBegin + rowsPerCore_;
```


```text
if (rowEnd > rows_) rowEnd = rows_;
```


设备侧入口读取分块参数后创建计算对象。倒平方根在设备上计算，并对稳定项进行合法性检查，确保分母始终为正。


```cpp
extern "C" __global__ __aicore__ void rmsnorm_baseline_kernel(
```


```text
GM_ADDR input, GM_ADDR weight, GM_ADDR output, GM_ADDR workspace, GM_ADDR tiling)
```


```text
{
```


```text
const __gm__ uint32_t *u = reinterpret_cast<const __gm__ uint32_t *>(tiling);
```


```text
const __gm__ float *f = reinterpret_cast<const __gm__ float *>(tiling);
```


```text
KernelRmsNormBaseline op;
```


```text
op.Init(input, weight, output, u[0], u[1], u[2], u[3], f[4], f[5]);
```


```text
op.Process();
```


```text
}
```


## 步骤二：编写RMSNorm核心计算核函数


本步把RMSNorm公式逐项落实为设备端代码，产出可独立核验的基础版核函数。


基础版按两次读取的方式实现：第一次读取输入并累加平方和，第二次读取输入和权重，完成缩放后写出结果。这样做并不追求速度，而是让公式与代码一一对应，便于检查数值是否正确。


## 核心设备端代码


```text
for (uint32_t row = rowBegin; row < rowEnd; ++row) {
```


```text
const uint32_t base = row * hidden_;
```


```text
float squareSum = 0.0f;
```


```text
for (uint32_t col = 0; col < hidden_; ++col) {
```


```text
const float x = inputGm_.GetValue(base + col);
```


```text
squareSum += x * x;
```


```text
}
```


```text
const float scale = RmsNormInvSqrtApprox(squareSum * invHidden_ + eps_);
```


```text
for (uint32_t col = 0; col < hidden_; ++col) {
```


```text
const float x = inputGm_.GetValue(base + col);
```


```text
const float w = weightGm_.GetValue(col);
```


```text
outputGm_.SetValue(base + col, x * scale * w);
```


```text
}
```


```text
}
```


## 步骤三：编写主机侧调用与算子注册


本步完成输入检查、设备内存管理、核函数启动和PyTorch算子注册，产出统一的上层调用入口。


基础版在框架侧提供统一调用入口。封装层负责检查输入形状和数据类型、准备分块参数、申请设备内存、启动内核并取回结果；上层Python测试只需按普通函数方式调用该算子。


```text
TORCH_CHECK(input.device().is_cpu(), "基础版wrapper expects CPU input tensor");
```


```text
TORCH_CHECK(input.scalar_type() == at::kFloat && weight.scalar_type() == at::kFloat);
```


```text
TORCH_CHECK(input.size(input.dim() - 1) == weight.size(0));
```


```text
auto inputContig = input.contiguous();
```


```text
auto weightContig = weight.contiguous();
```


```text
auto output = at::empty_like(inputContig);
```


```text
ACLRT_LAUNCH_KERNEL(rmsnorm_baseline_kernel)(coreNum, g_stream,
```


```text
inputD, weightD, outD, workspaceD, tilingD);
```


```text
CHECK_ACL(aclrtSynchronizeStream(g_stream));
```


```text
CHECK_ACL(aclrtMemcpy(output.data_ptr<float>(), inputBytes, outD, inputBytes,ACL_MEMCPY_DEVICE_TO_HOST));
```


```text
return output.view(input.sizes());
```


```text
TORCH_LIBRARY(rmsnorm_custom, m) {
```


```text
m.def("rms_norm(Tensor input, Tensor weight, float eps=1e-6) -> Tensor");
```


```text
}
```


```text
TORCH_LIBRARY_IMPL(rmsnorm_custom, CompositeExplicitAutograd, m) {
```


```text
m.impl("rms_norm", rmsnorm_baseline_npu);
```


```text
}
```


## 步骤四：构建算子并加载动态库


本步生成设备核函数库、框架注册库和独立测试程序，并确认运行时可以正确加载。


以下命令用于构建算子、生成调用库和运行测试。工具链、框架版本或硬件型号变化后，应重新构建，不宜直接复用其他环境生成的文件。


```bash
cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops/RmsNormBaselineExperiment
```


```bash
source /home/developer/Ascend/cann-8.5.2/set_env.sh
```


```bash
export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH
```


```bash
bash scripts/check_env.sh
```


```bash
bash scripts/build.sh
```


```bash
python3 tests/test_torch_op.py
```


## 步骤五：Golden数据与模型接入正确性验证


本步先以PyTorch参考实现进行单算子Golden对比，再用模型级调用检查权重、稳定项和替换位置。


正确性测试使用固定随机种子，覆盖一维、二维和三维输入。参考结果由PyTorch根据同一公式计算，再比较两者的最大误差、平均误差和是否满足允许误差范围。


```python
def reference(input_tensor, weight, eps):
```


```text
variance = input_tensor.pow(2).mean(dim=-1, keepdim=True)
```


```python
return input_tensor * torch.rsqrt(variance + eps) * weight
```


```text
golden = reference(input_tensor, weight, eps)
```


```python
actual = torch.ops.rmsnorm_custom.rms_norm(input_tensor, weight, eps)
```


```text
diff = (actual - golden).abs()
```


```python
ok = torch.allclose(actual, golden, atol=atol, rtol=rtol)
```


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">版本</th>
<th style="text-align:left;">单测覆盖结果</th>
<th style="text-align:left;">判定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">基础版</td>
<td style="text-align:left;">(1024,)、(128,1024)、(2,4,1024) 均PASS；最大绝对误差不超过7.15255737e-07</td>
<td style="text-align:left;">PASS</td>
</tr>
</tbody></table>


除单元测试外，工程还提供模型级对比测试，用于检查权重、稳定项和调用位置是否与原模型一致。模型级测试只验证接入正确性，不替代本节的单算子设备侧计时。


## 步骤六：性能计时、结果记录与分析


本步先在固定形状、预热次数和重复次数下统计纯设备侧内核时间，再使用CANN Profiling采集AI Core内部计算与数据搬运指标；两类结果共同构成优化实验的性能基线。


独立测试程序生成固定输入并计算参考结果。它先进行预热，再重复启动内核，并通过设备事件统计单次执行时间。该时间不包含Python调用、数据准备、结果比对和性能分析工具的开销。


```bash
cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops/RmsNormBaselineExperiment
```


```bash
source /home/developer/Ascend/cann-8.5.2/set_env.sh
```


```bash
export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH
```


```bash
out/bin/rmsnorm_baseline_standalone \
```


```text
--rows 128 --hidden 1024 --block-dim 8 --warmup 10 --repeat 50 --rounds 5 --eps 1e-6
```


```text
// standalone中的统计核心
```


```text
aclrtRecordEvent(start, stream);
```


```text
for (uint32_t i = 0; i < opt.repeat; ++i) {
```


```text
ACLRT_LAUNCH_KERNEL(rmsnorm_baseline_kernel)(blockDim, stream, ...);
```


```text
}
```


```text
aclrtRecordEvent(stop, stream);
```


```text
aclrtEventElapsedTime(&elapsedMs, start, stop);
```


```text
const double us = elapsedMs * 1000.0 / opt.repeat;
```


## Profiling采集与瓶颈分析


ACL Event只能给出一次内核启动的总体设备侧时间。为了判断时间主要消耗在标量计算、向量计算还是数据搬运，应在独立测试程序已经通过正确性验证后，使用CANN msprof采集AI Core运行数据。Profiling会引入额外开销，因此采集结果用于定位瓶颈，不直接作为最终性能数字。


```bash
cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops/RmsNormBaselineExperiment
```


```bash
source /home/developer/Ascend/cann-8.5.2/set_env.sh
```


```bash
export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH
```


```bash
mkdir -p profiling_output
```


```bash
msprof --application="./out/bin/rmsnorm_baseline_standalone --rows 128 --hidden 1024 --block-dim 8 --warmup 10 --repeat 50 --rounds 5 --eps 1e-6" --output=./profiling_output
```


采集完成后，应将Profiling报告与ACL Event结果交叉检查，重点关注以下指标：① aiv_scalar_time与aiv_vec_time的占比，用于判断逐元素标量循环是否成为主要瓶颈；② aic_mte_time及内存带宽，用于判断两次读取输入和细粒度GM访问造成的数据搬运开销；③ AI Core利用率与各核心任务分布，用于检查rowsPerCore划分是否均衡；④ kernel执行时间与事件计时是否处于同一量级。若基础版表现为scalar_time占比较高、向量单元利用率偏低且有效带宽远低于设备峰值，即可将“标量累加、重复读取和细粒度访存”确认为后续优化版需要解决的主要问题。


本步产出：profiling_output目录、关键指标记录以及瓶颈结论。验收时既要保留总体设备侧时间，也要说明主要耗时单元和数据搬运特征，不能仅以msprof采集时的端到端耗时评价算子性能。


## 实测结果与解释


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">版本</th>
<th style="text-align:left;">mean (us)</th>
<th style="text-align:left;">median (us)</th>
<th style="text-align:left;">min / max(us)</th>
<th style="text-align:left;">max_abs /mean_abs</th>
<th style="text-align:left;">正确性</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">基础版</td>
<td style="text-align:left;">491.313</td>
<td style="text-align:left;">491.275</td>
<td style="text-align:left;">491.262 / 491.403</td>
<td style="text-align:left;">7.15255737e-07 / 2.35556367e-08</td>
<td style="text-align:left;">PASS</td>
</tr>
</tbody></table>


本节数据来自项目测试日志。测试固定输入形状、计算核心数量、预热次数和重复次数，便于后续优化版在相同条件下进行比较。


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RmsNormBaselineExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RmsNormBaselineExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/rmsnorm_baseline_standalone --rows 128 --hidden 1024 --block-dim 8 --warmup 10 --repeat 50 --rounds 5 --eps 1e-6


# 常见问题与排查


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">现象</th>
<th style="text-align:left;">优先检查</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">torch.ops.rmsnorm_custom.rms_norm不存在</td>
<td style="text-align:left;">确认已bash scripts/build.sh，LD_LIBRARY_PATH包含out/lib，且register .so被load_torch_ops()加载</td>
</tr>
<tr>
<td style="text-align:left;">数值误差超过阈值</td>
<td style="text-align:left;">核对input最后一维、weight长度、FP32、eps、连续布局和tiling中invHidden</td>
</tr>
<tr>
<td style="text-align:left;">优化版直接报错</td>
<td style="text-align:left;">hidden是否为8的倍数；该限制来自对齐的GM↔UB DataCopy</td>
</tr>
<tr>
<td style="text-align:left;">性能异常</td>
<td style="text-align:left;">确认warmup后才计时；使用相同blockDim、shape、repeat、rounds；不要把wrapper/拷贝时间与standalone device时间混合</td>
</tr>
</tbody></table>


# 实验总结


本实验完成了RMSNorm基础版从数学公式、分块参数、AscendC内核、ACL/PyTorch注册到单元测试和独立性能验证的完整链路。所有性能结论均标明输入形状和计时范围；代码片段与RmsNormBaselineExperiment中的实际实现一致，可作为复现和进一步优化的起点。
